# CrewAI — Complex Agent

**Framework:** CrewAI  
**Level:** 03 — Complex  
**Model:** `gemini-2.0-flash`

### What's New vs Intermediate
| Feature | Intermediate | Complex |
|---|---|---|
| Agents | 2 | **3** (Planner + Researcher + Critic) |
| Tasks | 2 | **3 chained tasks** |
| Pattern | Sequential pipeline | **Plan → Research → Critique** |
| Output | Single `TravelBriefing` | **`TripReport` with ranked list** |
| Process | `sequential` | **`sequential` with multi-agent chaining** |

## Concept: Three-Agent Pipeline

```
┌────────────────────────────────────────────────────────────────┐
│                           CREW                                 │
│                                                                │
│  ┌──────────────┐    ┌──────────────┐    ┌──────────────────┐  │
│  │  Planner     │    │  Researcher  │    │  Critic          │  │
│  │  tools: none │    │  tools: 3    │    │  tools: none     │  │
│  └──────┬───────┘    └──────┬───────┘    └──────────────────┘  │
│         │                  │                       ▲           │
│  ┌──────▼───────┐    ┌──────▼───────┐    ┌──────────────────┐  │
│  │ Task 1:      │    │ Task 2:      │    │ Task 3:          │  │
│  │ Create plan  │──▶ │ Research     │──▶ │ Rank & critique  │  │
│  │ for cities   │    │ all cities   │    │ output_pydantic= │  │
│  └──────────────┘    └──────────────┘    │ TripReport       │  │
│       context=[]       context=[t1]      │ context=[t1, t2] │  │
│                                          └──────────────────┘  │
└────────────────────────────────────────────────────────────────┘
```

**CrewAI's Reflexion model:** Unlike LangGraph (retry loop via graph edge) or ADK (`score_report` tool), CrewAI achieves critique via a **dedicated Critic agent** whose task is to evaluate and synthesize. The Planner sets the strategy, the Researcher gathers data, and the Critic produces the final ranked output — each agent's output is the next agent's context.

**`context=[t1, t2]`** means the Critic task has access to both the planning output AND the research output — it sees the full chain.

## Setup

In [ ]:
import os
from pydantic import BaseModel, Field
from typing import List
from dotenv import load_dotenv

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your .env"
print("✓ Ready")

## Rich Output Schema — Ranked List

In [ ]:
class CityAnalysis(BaseModel):
    city: str
    weather_score: int = Field(ge=1, le=10, description="Weather score 1-10")
    safety_score: int = Field(ge=1, le=10, description="Safety score 1-10")
    summary: str = Field(description="One-sentence city summary")

class TripReport(BaseModel):
    ranked_cities: List[CityAnalysis] = Field(description="Cities ranked best to worst")
    top_recommendation: str = Field(description="The single best city to visit")
    reasoning: str = Field(description="Why the top city was chosen")

print("Output schema: TripReport with ranked CityAnalysis list")

## 3 Tools (for the Researcher only)

In [ ]:
@tool("Weather Scorer")
def get_weather(city: str) -> str:
    """Get weather and weather score (1-10) for a city. Input: city name."""
    data = {
        "london":    ("Cloudy",       12, 5), "new york":  ("Sunny",         22, 8),
        "bangalore": ("Rainy",        25, 6), "tokyo":     ("Clear",         18, 9),
        "paris":     ("Partly Cloudy",16, 7),
    }
    key = city.lower()
    if key in data:
        cond, temp, score = data[key]
        return f"{city}: {cond}, {temp}°C. Weather score: {score}/10"
    return f"No data for '{city}'."


@tool("Safety Advisor")
def get_travel_advisory(city: str) -> str:
    """Get safety advisory and safety score (1-10) for a city. Input: city name."""
    data = {
        "london":    ("Low",    "Routine precautions.",      8),
        "new york":  ("Low",    "Normal precautions.",       7),
        "bangalore": ("Medium", "Monsoon affects transport.", 6),
        "tokyo":     ("Low",    "Very safe city.",           10),
        "paris":     ("Low",    "Alert in crowded spots.",    8),
    }
    key = city.lower()
    if key in data:
        level, notes, score = data[key]
        return f"{city}: Advisory {level}. {notes} Safety score: {score}/10"
    return f"No data for '{city}'."


@tool("Time Checker")
def get_time(city: str) -> str:
    """Get current local time for a city. Input: city name."""
    times = {
        "london": "14:30 GMT", "new york": "09:30 EST",
        "bangalore": "20:00 IST", "tokyo": "22:30 JST", "paris": "15:30 CET",
    }
    key = city.lower()
    return f"{city}: {times.get(key, 'Unknown')}"


print("Tools ready: Weather Scorer, Safety Advisor, Time Checker")

## 3 Agents — Distinct Roles

The Planner has no tools — it only thinks. The Researcher uses all three tools. The Critic synthesizes — no tools needed, just analytical judgment.

In [ ]:
gemini = LLM(model="gemini/gemini-2.0-flash", temperature=0)

planner = Agent(
    role="Trip Planning Strategist",
    goal="Decompose travel comparison goals into a clear research plan with specific cities and evaluation criteria.",
    backstory=(
        "You are a strategic travel consultant. You receive travel goals and turn them into "
        "structured research plans — identifying cities to compare and the exact criteria to evaluate."
    ),
    tools=[],
    llm=gemini,
    verbose=False,
)

researcher = Agent(
    role="Travel Data Researcher",
    goal="Gather comprehensive weather, safety, and time data for every city in the plan.",
    backstory=(
        "You are a meticulous travel intelligence analyst. Given a research plan, you systematically "
        "collect weather scores, safety advisories, and local times. You never skip a city."
    ),
    tools=[get_weather, get_travel_advisory, get_time],
    llm=gemini,
    verbose=False,
    max_iter=8,
)

critic = Agent(
    role="Travel Report Editor",
    goal="Evaluate research data and produce a polished ranked travel report with clear justification.",
    backstory=(
        "You are a senior travel editor. You receive raw research and transform it into clear, "
        "ranked, actionable travel guides. Your rankings are always data-driven and clearly justified."
    ),
    tools=[],
    llm=gemini,
    verbose=False,
)

print(f"Agents: {planner.role} | {researcher.role} | {critic.role}")

## 3 Tasks with Full Context Chain

In [ ]:
def build_crew(goal: str, cities: list) -> Crew:
    cities_str = ", ".join(cities)

    planning_task = Task(
        description=(
            f"Create a research plan for comparing: {cities_str}.\n"
            f"Goal: {goal}\n"
            "Output: numbered plan listing each city + data to collect."
        ),
        expected_output=f"Numbered plan for {cities_str}: weather, safety, time per city.",
        agent=planner,
    )

    research_task = Task(
        description=(
            f"Execute the research plan. For each city ({cities_str}), collect:\n"
            "- Weather conditions + score\n- Safety advisory + score\n- Local time\n"
            "Use all three tools for EVERY city."
        ),
        expected_output=f"Raw data for all {len(cities)} cities with numeric scores.",
        agent=researcher,
        context=[planning_task],
    )

    critique_task = Task(
        description=(
            f"Using the research data, produce a ranked travel comparison for: {cities_str}.\n"
            f"Original goal: {goal}\n"
            "Requirements:\n"
            "1. Rank all cities best to worst with score justification\n"
            "2. Provide a clear top recommendation\n"
            "3. Base reasoning on weather + safety scores\n"
            "Fill every field of the TripReport schema."
        ),
        expected_output="A TripReport with ranked_cities, top_recommendation, and reasoning.",
        agent=critic,
        context=[planning_task, research_task],  # sees BOTH previous tasks
        output_pydantic=TripReport,
    )

    return Crew(
        agents=[planner, researcher, critic],
        tasks=[planning_task, research_task, critique_task],
        process=Process.sequential,
        verbose=True,
    )

print("Crew factory ready")

## Run

In [ ]:
goal = "I want the safest city with the best weather for a week-long trip."
cities = ["Tokyo", "Paris", "Bangalore"]

crew = build_crew(goal, cities)
result = crew.kickoff()

In [ ]:
# Display structured output
report: TripReport = result.pydantic

if report:
    print("\n" + "="*60)
    print("TRIP REPORT")
    print("="*60)
    print("\nRANKING:")
    for i, city in enumerate(report.ranked_cities, 1):
        bar_w = "█" * city.weather_score + "░" * (10 - city.weather_score)
        bar_s = "█" * city.safety_score + "░" * (10 - city.safety_score)
        print(f"  {i}. {city.city}")
        print(f"     Weather : {bar_w} {city.weather_score}/10")
        print(f"     Safety  : {bar_s} {city.safety_score}/10")
        print(f"     Summary : {city.summary}")
    print(f"\nTop Pick  : {report.top_recommendation}")
    print(f"Why       : {report.reasoning}")
else:
    print("Structured output not available. Raw output:")
    print(result.raw)

In [ ]:
# Try different criteria — same agents, different goal
goal2 = "I love warm, rainy weather and don't mind medium safety advisories."
crew2 = build_crew(goal2, ["Tokyo", "Paris", "Bangalore"])
result2 = crew2.kickoff()

report2 = result2.pydantic
if report2:
    print(f"\nFor goal: '{goal2}'")
    print(f"Top Pick: {report2.top_recommendation}")
    print(f"Why: {report2.reasoning}")

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| 3 agents with distinct roles | Separation of concerns: strategy / data / judgment |
| `context=[t1, t2]` on critique task | Critic sees the full chain — plan + raw data + its own judgment |
| `output_pydantic=TripReport` | Structured ranked list — not just a text summary |
| `max_iter=8` on researcher | Prevents infinite tool-calling loops |
| Changing the goal changes the ranking | Critic interprets the goal — same data, different weighting |

### All 4 Frameworks — Complete Comparison

| Level | ADK | LangChain | LangGraph | CrewAI |
|---|---|---|---|---|
| **Simple** | `Agent` + plain functions | `create_tool_calling_agent` + `AgentExecutor` | `StateGraph` + `ToolNode` | `Agent` + `Task` + `Crew` |
| **Intermediate** | `output_schema` + session memory | `RunnableWithMessageHistory` | `MemorySaver` + rich state | Multi-agent + `context` |
| **Complex** | `score_report` tool + instruction | Plan-and-Execute + Critic chain | Reflexion graph + 6 nodes | 3-agent pipeline + `TripReport` |

### What's Next
- [02-Architectures/](../../02-Architectures/README.md) — multi-agent coordination patterns (Sequential, Parallel, Hierarchical...)
- [ADK vertex-ai-real-world/](../../ADK/vertex-ai-real-world/) — run the complex agent on Vertex AI with a real GCP project